# ASL (American Sign Language) Model Training

Train AI model to recognize 5 ASL signs using MediaPipe + scikit-learn:
- HELLO
- THANK YOU
- HELP
- YES
- NO

In [ ]:
# Install required libraries
!pip install mediapipe opencv-python scikit-learn numpy

In [ ]:
# Import libraries
import cv2
import numpy as np
import mediapipe as mp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pickle
import os

In [ ]:
# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)

In [ ]:
# Step 1: Upload your sign language videos to Colab
# You can upload via the file browser or mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Organize videos like this:
# /content/drive/MyDrive/asl_dataset/
#   ├── HELLO/
#   ├── THANK_YOU/
#   ├── HELP/
#   ├── YES/
#   └── NO/

In [ ]:
# Step 2: Extract hand landmarks from videos
def extract_landmarks(video_path):
    cap = cv2.VideoCapture(video_path)
    landmarks_list = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Process with MediaPipe
        results = hands.process(frame_rgb)
        
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # Extract 21 landmarks (x, y, z) = 63 features
                landmarks = []
                for landmark in hand_landmarks.landmark:
                    landmarks.extend([landmark.x, landmark.y, landmark.z])
                landmarks_list.append(landmarks)
    
    cap.release()
    return landmarks_list

# Extract features from all videos
data = []
labels = []
signs = ['HELLO', 'THANK_YOU', 'HELP', 'YES', 'NO']

for sign in signs:
    sign_folder = f'/content/drive/MyDrive/asl_dataset/{sign}'
    for video_file in os.listdir(sign_folder):
        video_path = os.path.join(sign_folder, video_file)
        landmarks = extract_landmarks(video_path)
        data.extend(landmarks)
        labels.extend([sign] * len(landmarks))

print(f'Extracted {len(data)} samples')

In [ ]:
# Step 3: Train Random Forest classifier
X = np.array(data)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f'Model accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Step 4: Save the trained model
with open('/content/drive/MyDrive/asl_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Model saved as asl_model.pkl')

In [ ]:
# Step 5: Test the model on a sample video
test_video = '/content/drive/MyDrive/test_video.mp4'
test_landmarks = extract_landmarks(test_video)

if test_landmarks:
    predictions = model.predict(test_landmarks)
    print(f'Predicted signs: {predictions}')